# Orochi Model Testing Notebook

This notebook demonstrates how to:
1. Load the Orochi 3D Mamba model
2. Load pretrained weights
3. Run inference on a test image
4. Visualize results

**Pretrained checkpoints:**
- `pretrained_checkpoints/mamba_fm_3d.pth.tar`
- `pretrained_checkpoints/MambaULight_epoch_99_loss_-0.0624.pth.tar`

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add package to path if needed
package_root = Path.cwd()
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace

# Import Orochi components
from orochi.models import MambaEncoderHeria, reg_decoder
from orochi.losses import get_loss_function

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")

## Model Configuration

Configure the model for 3D processing

In [ ]:
# Create configuration for 3D model
config = SimpleNamespace(
    # Model architecture
    dimensions=3,                    # 3D processing
    img_size=(160, 192, 224),       # Standard brain MRI size
    in_chans=2,                      # 2 channels for registration (moving + fixed)
    embed_dim=96,                    # Embedding dimension
    depths=[2, 2, 2, 2],            # Depth of each stage
    num_heads=[3, 6, 12, 24],       # Number of attention heads per stage
    window_size=(5, 6, 7),          # 3D window size
    patch_size=4,                    # Patch size for embedding
    drop_rate=0.0,                   # Dropout rate
    drop_path_rate=0.1,              # Stochastic depth rate
    use_checkpoint=False,            # Gradient checkpointing (for memory)
)

print("Model Configuration:")
print(f"  Dimensions: {config.dimensions}D")
print(f"  Input size: {config.img_size}")
print(f"  Input channels: {config.in_chans}")
print(f"  Embed dim: {config.embed_dim}")
print(f"  Depths: {config.depths}")
print(f"  Window size: {config.window_size}")

## Initialize Model

Create the encoder and decoder

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create encoder
print("\nInitializing MambaEncoderHeria...")
encoder = MambaEncoderHeria(config).to(device)

# Create decoder (for registration)
print("Initializing registration decoder...")
decoder = reg_decoder(config).to(device)

# Count parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
decoder_params = sum(p.numel() for p in decoder.parameters())
total_params = encoder_params + decoder_params

print(f"\n✓ Model initialized successfully!")
print(f"  Encoder parameters: {encoder_params:,}")
print(f"  Decoder parameters: {decoder_params:,}")
print(f"  Total parameters: {total_params:,}")

## Load Pretrained Weights

Load weights from checkpoint file

In [ ]:
# Available checkpoints
checkpoint_dir = Path('pretrained_checkpoints')
checkpoints = [
    'mamba_fm_3d.pth.tar',
    'MambaULight_epoch_99_loss_-0.0624.pth.tar'
]

# Choose checkpoint
checkpoint_name = checkpoints[0]  # Change index to use different checkpoint
checkpoint_path = checkpoint_dir / checkpoint_name

if checkpoint_path.exists():
    print(f"Loading checkpoint: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Check checkpoint structure
    print(f"\nCheckpoint keys: {list(checkpoint.keys())}")
    
    # Load state dict (adjust key based on checkpoint structure)
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint  # Assume checkpoint is the state dict itself
    
    # Load encoder weights
    try:
        # Try to load directly
        encoder.load_state_dict(state_dict, strict=False)
        print("\n✓ Weights loaded successfully!")
    except Exception as e:
        print(f"\n⚠ Warning: Could not load all weights: {e}")
        print("Attempting to load matching keys only...")
        
        # Load only matching keys
        model_dict = encoder.state_dict()
        pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.size() == model_dict[k].size()}
        model_dict.update(pretrained_dict)
        encoder.load_state_dict(model_dict)
        
        print(f"✓ Loaded {len(pretrained_dict)}/{len(model_dict)} parameters")
    
    # Check if checkpoint has additional info
    if 'epoch' in checkpoint:
        print(f"  Checkpoint epoch: {checkpoint['epoch']}")
    if 'loss' in checkpoint:
        print(f"  Checkpoint loss: {checkpoint['loss']}")
        
else:
    print(f"⚠ Checkpoint not found: {checkpoint_path}")
    print(f"Available checkpoints: {list(checkpoint_dir.glob('*.pth*')) if checkpoint_dir.exists() else 'Directory not found'}")
    print("\nContinuing with random initialization...")

## Create Test Data

Generate synthetic test volumes or load real data

In [ ]:
# Option 1: Create synthetic test data
print("Creating synthetic test data...")

# Create moving and fixed images
batch_size = 1
depth, height, width = 160, 192, 224

# Generate random volumes (in practice, load real MRI data)
moving = torch.randn(batch_size, 1, depth, height, width, device=device)
fixed = torch.randn(batch_size, 1, depth, height, width, device=device)

# Normalize to [0, 1]
moving = (moving - moving.min()) / (moving.max() - moving.min())
fixed = (fixed - fixed.min()) / (fixed.max() - fixed.min())

print(f"✓ Test data created:")
print(f"  Moving image shape: {moving.shape}")
print(f"  Fixed image shape: {fixed.shape}")
print(f"  Value range: [{moving.min():.3f}, {moving.max():.3f}]")

# Option 2: Load real data (uncomment if you have data)
# import nibabel as nib
# moving_nii = nib.load('path/to/moving.nii.gz')
# fixed_nii = nib.load('path/to/fixed.nii.gz')
# moving = torch.from_numpy(moving_nii.get_fdata()).unsqueeze(0).unsqueeze(0).float().to(device)
# fixed = torch.from_numpy(fixed_nii.get_fdata()).unsqueeze(0).unsqueeze(0).float().to(device)

## Run Inference

Perform forward pass through the model

In [ ]:
# Set model to eval mode
encoder.eval()
decoder.eval()

print("Running inference...")

with torch.no_grad():
    # Concatenate moving and fixed images
    combined = torch.cat([moving, fixed], dim=1)  # [B, 2, D, H, W]
    print(f"  Input shape: {combined.shape}")
    
    # Encoder forward pass
    features = encoder(combined)
    print(f"  Number of feature maps: {len(features)}")
    for i, feat in enumerate(features):
        print(f"    Level {i}: {feat.shape}")
    
    # Decoder forward pass (produces flow field)
    flow = decoder(features)
    print(f"  Output flow shape: {flow.shape}")
    print(f"  Flow value range: [{flow.min():.3f}, {flow.max():.3f}]")

print("\n✓ Inference completed successfully!")

## Apply Deformation

Warp the moving image using the predicted flow field

In [ ]:
def warp_3d(img, flow):
    """
    Warp a 3D image using a flow field.
    
    Args:
        img: Image tensor [B, C, D, H, W]
        flow: Flow field [B, 3, D, H, W] (displacement in z, y, x)
    
    Returns:
        Warped image [B, C, D, H, W]
    """
    B, C, D, H, W = img.shape
    
    # Create sampling grid
    vectors = [torch.arange(0, s, device=img.device) for s in [D, H, W]]
    grids = torch.meshgrid(vectors, indexing='ij')
    grid = torch.stack(grids)  # [3, D, H, W]
    grid = grid.unsqueeze(0).float()  # [1, 3, D, H, W]
    
    # Add flow to grid
    new_grid = grid + flow
    
    # Normalize grid to [-1, 1]
    for i in range(3):
        new_grid[:, i, ...] = 2 * (new_grid[:, i, ...] / (img.shape[i+2] - 1)) - 1
    
    # Permute to [B, D, H, W, 3]
    new_grid = new_grid.permute(0, 2, 3, 4, 1)
    
    # Sample
    warped = nn.functional.grid_sample(
        img, new_grid,
        mode='bilinear',
        padding_mode='border',
        align_corners=True
    )
    
    return warped

# Warp moving image
with torch.no_grad():
    warped = warp_3d(moving, flow)

print(f"✓ Image warped successfully!")
print(f"  Warped image shape: {warped.shape}")
print(f"  Value range: [{warped.min():.3f}, {warped.max():.3f}]")

## Compute Metrics

Evaluate registration quality

In [ ]:
from orochi.metrics import get_metric

# Get metric functions
mse_fn = get_metric('mse')
mae_fn = get_metric('mae')

with torch.no_grad():
    # Compute metrics before registration
    mse_before = mse_fn(moving.cpu(), fixed.cpu())
    mae_before = mae_fn(moving.cpu(), fixed.cpu())
    
    # Compute metrics after registration
    mse_after = mse_fn(warped.cpu(), fixed.cpu())
    mae_after = mae_fn(warped.cpu(), fixed.cpu())

print("Registration Metrics:")
print(f"\n  Before Registration:")
print(f"    MSE: {mse_before:.6f}")
print(f"    MAE: {mae_before:.6f}")
print(f"\n  After Registration:")
print(f"    MSE: {mse_after:.6f}")
print(f"    MAE: {mae_after:.6f}")
print(f"\n  Improvement:")
print(f"    MSE: {((mse_before - mse_after) / mse_before * 100):.2f}%")
print(f"    MAE: {((mae_before - mae_after) / mae_before * 100):.2f}%")

## Visualize Results

Display middle slices of volumes

In [ ]:
def show_slices(moving, fixed, warped, flow, slice_idx=None):
    """
    Show middle slices of 3D volumes.
    """
    if slice_idx is None:
        slice_idx = moving.shape[2] // 2  # Middle slice
    
    # Extract slices
    moving_slice = moving[0, 0, slice_idx].cpu().numpy()
    fixed_slice = fixed[0, 0, slice_idx].cpu().numpy()
    warped_slice = warped[0, 0, slice_idx].cpu().numpy()
    
    # Flow magnitude
    flow_magnitude = torch.sqrt((flow[0] ** 2).sum(dim=0))[slice_idx].cpu().numpy()
    
    # Difference maps
    diff_before = np.abs(moving_slice - fixed_slice)
    diff_after = np.abs(warped_slice - fixed_slice)
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Row 1: Images
    axes[0, 0].imshow(moving_slice, cmap='gray')
    axes[0, 0].set_title(f'Moving Image (Slice {slice_idx})')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(fixed_slice, cmap='gray')
    axes[0, 1].set_title(f'Fixed Image (Slice {slice_idx})')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(warped_slice, cmap='gray')
    axes[0, 2].set_title(f'Warped Image (Slice {slice_idx})')
    axes[0, 2].axis('off')
    
    # Row 2: Differences and flow
    im1 = axes[1, 0].imshow(diff_before, cmap='hot')
    axes[1, 0].set_title('Difference Before')
    axes[1, 0].axis('off')
    plt.colorbar(im1, ax=axes[1, 0], fraction=0.046)
    
    im2 = axes[1, 1].imshow(diff_after, cmap='hot')
    axes[1, 1].set_title('Difference After')
    axes[1, 1].axis('off')
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)
    
    im3 = axes[1, 2].imshow(flow_magnitude, cmap='viridis')
    axes[1, 2].set_title('Flow Magnitude')
    axes[1, 2].axis('off')
    plt.colorbar(im3, ax=axes[1, 2], fraction=0.046)
    
    plt.tight_layout()
    plt.show()

# Visualize
show_slices(moving, fixed, warped, flow)

## Save Results

Save the warped image and flow field

In [ ]:
# Create output directory
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

# Save tensors
torch.save({
    'moving': moving.cpu(),
    'fixed': fixed.cpu(),
    'warped': warped.cpu(),
    'flow': flow.cpu(),
    'config': vars(config),
}, output_dir / 'registration_results.pth')

print(f"✓ Results saved to {output_dir}/registration_results.pth")

# Optionally save as NIfTI (requires nibabel)
# import nibabel as nib
# warped_nii = nib.Nifti1Image(warped[0, 0].cpu().numpy(), affine=np.eye(4))
# nib.save(warped_nii, output_dir / 'warped.nii.gz')
# print(f"✓ Warped image saved as NIfTI")

## Model Summary

Print model architecture details

In [ ]:
print("\n" + "="*70)
print("MODEL SUMMARY")
print("="*70)

print(f"\nEncoder Architecture:")
print(f"  Type: MambaEncoderHeria (3D)")
print(f"  Input size: {config.img_size}")
print(f"  Patch size: {config.patch_size}")
print(f"  Embed dim: {config.embed_dim}")
print(f"  Depths: {config.depths}")
print(f"  Heads: {config.num_heads}")
print(f"  Parameters: {encoder_params:,}")

print(f"\nDecoder Architecture:")
print(f"  Type: Registration Decoder (3D)")
print(f"  Output: 3D displacement field")
print(f"  Parameters: {decoder_params:,}")

print(f"\nTotal Model:")
print(f"  Parameters: {total_params:,}")
print(f"  Size: {total_params * 4 / 1024**2:.2f} MB (fp32)")

print("\n" + "="*70)

## Next Steps

**To use this notebook:**

1. **With pretrained weights:**
   - Place checkpoint files in `pretrained_checkpoints/` directory
   - Run all cells

2. **With your own data:**
   - Modify the "Create Test Data" cell to load your images
   - Ensure images are normalized to [0, 1]
   - Adjust `config.img_size` to match your data

3. **For different tasks:**
   - Change decoder: `SR_decoder`, `fus_decoder`, `IR_decoder`
   - Adjust `config.in_chans` for different inputs
   - Modify loss functions for training

**Documentation:**
- See `docs/QUICK_START_GUIDE.md` for more examples
- See `docs/MAMBA_CONSOLIDATION_ANALYSIS.md` for architecture details
- See `IMPORT_FIX_SUMMARY.md` for import reference